In [ ]:
%pip install psycopg2-binary 

In [ ]:
import psycopg2
import torch
from PIL import Image
from transformers import AutoModel, AutoProcessor

model = AutoModel.from_pretrained('Marqo/marqo-fashionSigLIP', trust_remote_code=True)
processor = AutoProcessor.from_pretrained('Marqo/marqo-fashionSigLIP', trust_remote_code=True)

# Database connection configuration
db_config = {
    'dbname': 'image-search',
    'user': 'postgres',
    'password': 'test-postgres',
    'host': '127.0.0.1',
    'port': '5432'
}


# Connect to PostgreSQL database
def connect_to_database(config):
    """Connect to PostgreSQL database."""
    try:
        conn = psycopg2.connect(**config)
        # print("Database connection established.")
        return conn
    except Exception as e:
        print(f"Database connection failed: {e}")
        raise


def input_i_processor(image):
    """Process input image to generate embeddings."""
    processed = processor(images=image, padding=True, truncation=True, return_tensors="pt")

    with torch.no_grad():
        image_features = model.get_image_features(processed['pixel_values'],
                                                  normalize=True).cpu().numpy().flatten().tolist()
    return image_features


# Perform a similarity search
def perform_search(conn, embedding_query, top_k, column_name):
    cur = conn.cursor()
    # print(f"Performing search on column '{column_name}' with top_k = {top_k}")
    try:
        cur.execute(f"""
            SELECT image_name, {column_name} <=> %s::vector AS distance
            FROM fashion_clip_test
            ORDER BY distance ASC
            LIMIT %s;
        """, (embedding_query, top_k))

        similar_items = cur.fetchall()

        # print(f"Found {len(similar_items)} similar items.")

        # for item in similar_items:
        #     print("Similar item:", item)
        #
        # return similar_items

        # Extract and return only the image names from the results
        similar_items = [result[0] for result in similar_items]

        return similar_items

    except Exception as e:
        print(f"Error performing search: {e}")
        return []
    finally:
        cur.close()


# Handling image-to-image queries
def image_to_image_search(conn, query_image, top_k):
    # print("Running image-to-image search...")
    query_image_embedding = input_i_processor(query_image)
    return perform_search(conn, query_image_embedding, top_k, "image_embedding")


# main function
def main():
    try:
        conn = connect_to_database(db_config)

        query_image_path = "/Users/Tommy/AI/fashionCLIP/clothing-images/061/0619350002.jpg"
        query_image = Image.open(query_image_path)
        # print("Running image-to-image query...")
        image_results = image_to_image_search(conn, query_image, top_k=10)

        print("image_to_image_search_results: ", image_results)

    except Exception as e:
        print(f"Error in main processing: {e}")
    finally:
        if conn:
            conn.close()
            # print("Database connection closed.")

In [ ]:
if __name__ == "__main__":
    main()

In [ ]:
precision:70%
['0495733003.jpg', v
 '0559956011.jpg', v
 '0560030004.jpg', v
 '0503502003.jpg', v
 '0497787004.jpg', v
 '0544054005.jpg', v
 '0559757003.jpg', 
 '0795836009.jpg', v
 '0795836002.jpg'
 '0795836002.jpg']